# 15 · Risk decomposition & market regimes

Where does portfolio risk actually come from, and which market state are we in? Euler decomposition splits volatility into additive per-asset parts; PCA finds the common factors; regime labels are **causal** (point-in-time).

In [1]:
import numpy as np
import pandas as pd
from quantkit import risk as RK, regime as RG, visualization as V

rng = np.random.default_rng(3)
dates = pd.bdate_range('2016-01-01', periods=750)
assets = [f'A{i:02d}' for i in range(8)]
mkt = rng.standard_normal(len(dates)) * 0.01  # common factor
rets = pd.DataFrame(
    {a: mkt + 0.006 * rng.standard_normal(len(dates)) for a in assets},
    index=dates,
)
cov = rets.cov()
print('built returns:', rets.shape)

built returns: (750, 8)


## Euler volatility decomposition

`component_risk` sums exactly to `portfolio_vol` — the parts add up to the whole.

In [2]:
w = pd.Series(1.0 / len(assets), index=assets)
comp = RK.component_risk(w, cov)
print('portfolio vol :', round(RK.portfolio_vol(w, cov), 5))
print('sum of parts  :', round(comp.sum(), 5))
comp.round(5)

portfolio vol : 0.01029
sum of parts  : 0.01029


A00    0.00129
A01    0.00126
A02    0.00130
A03    0.00130
A04    0.00125
A05    0.00130
A06    0.00129
A07    0.00131
dtype: float64

## PCA risk factors

The first eigenvector is the market direction; `effective_n_bets` is the inverse-HHI of explained variance — how many independent bets the book really holds.

In [3]:
import plotly.graph_objects as go
pca = RK.pca_factors(rets, n_components=3)
print('explained variance ratio:', pca.explained_variance_ratio.round(3).to_dict())
print('effective n bets        :', round(RK.effective_n_bets(rets), 2))
V.factor_heatmap(pca.loadings, title='PCA loadings (assets x factors)')

explained variance ratio: {'PC1': 0.767, 'PC2': 0.038, 'PC3': 0.037}
effective n bets        : 1.68


## Causal market regimes

`vol_regime` buckets trailing realized vol against **expanding** (past-only) quantiles, so the label at t never moves when future data arrives.

In [4]:
mret = rets.mean(axis=1)
vol_reg = RG.vol_regime(mret, lookback=21, n_states=3, min_history=126)
summary = RG.regime_summary(mret, vol_reg)
print(summary.round(3).to_string())
eq = (1 + mret).cumprod()
fig = go.Figure()
fig.add_trace(go.Scatter(x=eq.index, y=eq.to_numpy(), name='equity', line={'color':'#111'}))
fig.add_trace(go.Scatter(x=vol_reg.index, y=vol_reg.to_numpy(), name='vol regime',
                         yaxis='y2', mode='lines', line={'color':'#E45756'}))
fig.update_layout(template='plotly_white', height=380, title='Equity vs vol regime',
                  yaxis2={'overlaying':'y','side':'right','title':'regime'})
fig

        count   mean    vol  sharpe
regime                             
0.0     289.0  0.212  0.138   1.534
1.0     176.0  0.092  0.160   0.575
2.0     159.0  0.206  0.192   1.078


**Takeaway.** Risk is rarely where weights suggest — PCA shows a single factor can dominate; regime labels let a backtest condition on state without look-ahead.